# Financial Blockchain Capstone: African Remittances Payment Prototype



## 1. Imports and hashing


In [29]:
from __future__ import annotations  # Allows type annotations to be evaluated lazily, which helps when annotations refer to types defined later.

import hashlib  # Provides hashing algorithms, including SHA-256
import json  # Provides functions for converting Python objects into JSON strings.
import time  
from copy import deepcopy  # Used to create a deepcopy not a shallow copy
from dataclasses import dataclass, field  # Imports tools for defining data classes and their fields
from typing import Any  # Type hint that allows a variable to hold values of different types.

DIFFICULTY = 2  # Sets the proof-of-work difficulty to 2; a valid hash must begin with two zeros.


def sha256_hex(data: bytes) -> str:  # Defines a function that accepts bytes and returns a hexadecimal hash as a string.
    return hashlib.sha256(data).hexdigest()  # Calculates the SHA-256 hash of the input bytes and returns it as a hexadecimal string.


def canonical(obj: object) -> str:  # Defines a function that accepts a Python object and converts it into a consistent JSON string.
    return json.dumps(obj, sort_keys=True, separators=(',', ':'))  # Sorts dictionary keys removes variances


print('DIFFICULTY =', DIFFICULTY)  # Displays the current proof-of-work difficulty setting.
print('sha256_hex(b"mZAR") =', sha256_hex(b'mZAR')[:16], '...')  # displays the first 16 hexadecimal characters

DIFFICULTY = 2
sha256_hex(b"mZAR") = d3b93a230d27eaa0 ...


## 2. wallets (demo keys)



In [30]:
@dataclass  # Automatically generates common class methods, such as __init__.
class Wallet:  # Defines a wallet class.
    label: str  # Stores the wallet's label.
    sk: str = ''  # Stores the secret key.
    pk: str = ''  # Stores the public key.
    address: str = ''  # Stores the wallet address.

    def __post_init__(self) -> None:  # Runs after the wallet is initialised.
        self.sk = sha256_hex(f'toy-sk-{self.label}'.encode())  # Generates a toy secret key.
        self.pk = sha256_hex(f'toy-pk-{self.sk}'.encode())  # Generates a toy public key.
        self.address = self.pk[:40]  # Uses the first 40 characters as the address.


def create_wallets() -> tuple[Wallet, Wallet, Wallet]:  # Defines a function that creates three wallets.
    """Return sender, recipient, and optional reserve agent."""  
    return Wallet('Sender'), Wallet('Recipient'), Wallet('ReserveAgent')  # Creates and returns the three wallets.


sender, recipient, reserve = create_wallets()  # Assigns each returned wallet to a variable.
print('Sender   ', sender.address)  # Displays the sender's address.
print('Recipient', recipient.address)  # Displays the recipient's address.
print('Reserve  ', reserve.address)  # Displays the reserve agent's address.

Sender    cc6154ce75ed89ab8dea0b12e45866c988f513ef
Recipient 56f62e486f82191a14f9f6ecaaa5329d0addfd02
Reserve   28312b71e08138db7b7cb1ce7f14255cf9ea71a7


## 3. Reserve ledger — the 1:1 invariant


In [31]:
@dataclass
class ReserveLedger:
    """Toy vault: vault_fiat should always equal circulating mZAR."""

    vault_fiat: int = 0
    circulating: int = 0
    balances: dict[str, int] = field(default_factory=dict)  # Tracks each address's mZAR balance.
    receipts: list[dict] = field(default_factory=list)  # Stores off-ramp receipts.

    def invariant_ok(self) -> bool:
        return self.vault_fiat == self.circulating  # Checks that the reserve equals circulating mZAR.

    def onramp(self, address: str, amount: int) -> None:
        if amount <= 0:
            raise ValueError('amount must be positive')
        self.vault_fiat += amount  # Increases the fiat reserve.
        self.circulating += amount  # Issues the equivalent amount of mZAR.
        self.balances[address] = self.balances.get(address, 0) + amount  # Credits the user's wallet.

    def apply_transfer(self, sender_addr: str, recipient_addr: str, amount: int) -> None:
        if self.balances.get(sender_addr, 0) < amount:
            raise ValueError('insufficient balance')
        self.balances[sender_addr] -= amount  # Deducts mZAR from the sender.
        self.balances[recipient_addr] = self.balances.get(recipient_addr, 0) + amount  # Credits the recipient.

    def offramp(self, address: str, amount: int) -> dict:
        if self.balances.get(address, 0) < amount:
            raise ValueError('insufficient balance')
        self.balances[address] -= amount  # Deducts the redeemed mZAR from the user's wallet.
        self.circulating -= amount  # Burns the redeemed tokens.
        self.vault_fiat -= amount  # Reduces the simulated reserve after redemption.
        receipt = {
            'receipt_id': sha256_hex(f'{address}:{amount}:{time.time()}'.encode())[:16],  # Generates a short receipt identifier.
            'address': address,
            'paid_local': amount,
        }
        self.receipts.append(receipt)  # Saves the receipt in the ledger.
        return receipt


# Smoke: starter test_onramp_invariant
_led = ReserveLedger()
_led.onramp('addrA', 5000)
assert _led.invariant_ok()  # Confirms the reserve invariant holds after the on-ramp.
assert _led.balances['addrA'] == 5000
assert _led.vault_fiat == 5000
print('ReserveLedger smoke OK — invariant', _led.invariant_ok())

ReserveLedger smoke OK — invariant True


## 4. Sign / verify (toy scheme — replace with ECDSA)



In [32]:
def sign(payload: dict, sk: str) -> str:
    # Excludes the signature to create a consistent payload for signing.
    body = canonical({k: payload[k] for k in payload if k != 'signature'})
    return sha256_hex(f'{sk}:{body}'.encode())  # Generates a toy signature using the secret key and payload.


def verify_tx(tx: dict, wallet: Wallet) -> bool:
    return tx.get('signature') == sign(tx, wallet.sk)  # Checks whether the signature matches the expected value.


def build_transfer(frm: Wallet, to_addr: str, amount: int, asset: str = 'mZAR') -> dict:
    tx = {
        'sender': frm.address,
        'recipient': to_addr,
        'amount': amount,
        'timestamp': int(time.time()),
        'asset': asset,
    }
    tx['signature'] = sign(tx, frm.sk)  # Signs the transaction using the sender's secret key.
    return tx


sample = build_transfer(sender, recipient.address, 10)
assert verify_tx(sample, sender)  # Confirms the original transaction passes verification.
tampered = dict(sample)
tampered['amount'] = 9999
assert not verify_tx(tampered, sender)  # Confirms changing the amount invalidates the signature.
print('sign/verify OK; tamper rejected')

sign/verify OK; tamper rejected


## 5. Chain helpers — mempool, mining, verification



In [33]:
def block_hash(block: dict) -> str:
    payload = {k: block[k] for k in ('index', 'timestamp', 'transactions', 'previous_hash', 'nonce')}
    return sha256_hex(canonical(payload).encode())


def mine_block(
    transactions: list,
    previous_hash: str,
    index: int,
    difficulty: int = DIFFICULTY,
) -> dict:
    prefix = '0' * difficulty
    nonce = 0
    timestamp = int(time.time())
    while True:
        cand = {
            'index': index,
            'timestamp': timestamp,
            'transactions': transactions,
            'previous_hash': previous_hash,
            'nonce': nonce,
        }
        h = block_hash(cand)
        if h.startswith(prefix):
            cand['hash'] = h
            return cand
        nonce += 1


def verify_chain(chain: list[dict]) -> bool:
    for i, block in enumerate(chain):
        if block_hash(block) != block['hash']:
            return False
        if i and block['previous_hash'] != chain[i - 1]['hash']:
            return False
    return True


def make_genesis() -> dict:
    return mine_block([{'note': 'genesis'}], '0' * 64, 0)


genesis = make_genesis()
assert verify_chain([genesis])
print('Genesis mined index=', genesis['index'], 'hash=', genesis['hash'][:16], '...')
print('verify_chain OK')

Genesis mined index= 0 hash= 00a5c12dbd1945f8 ...
verify_chain OK


## 6. Simulated on-ramp

Corridor step 1: Sender deposits **5\,000** local units → mint **5\,000 mZAR**. Assert the invariant immediately.

In [34]:
ledger = ReserveLedger()
chain: list[dict] = [make_genesis()]
mempool: list[dict] = []

LOCAL_DEPOSIT = 5000
TRANSFER_AMOUNT = 3500

ledger.onramp(sender.address, LOCAL_DEPOSIT)
assert ledger.invariant_ok()

print('ON-RAMP')
print('  vault_fiat   =', ledger.vault_fiat)
print('  circulating  =', ledger.circulating)
print('  sender bal   =', ledger.balances[sender.address])
print('  invariant_ok =', ledger.invariant_ok())

ON-RAMP
  vault_fiat   = 5000
  circulating  = 5000
  sender bal   = 5000
  invariant_ok = True


## 7. On-chain transfer + mine

Build a signed 3500 mZAR payment, accept into the mempool only if signature and balance check pass, then mine. Apply the transfer on the ledger **after** inclusion (vault stays 5\,000).

In [35]:
WALLETS_BY_ADDR = {
    sender.address: sender,
    recipient.address: recipient,
    reserve.address: reserve,
}


def accept_tx(tx: dict, led: ReserveLedger, pool: list[dict]) -> bool:
    """Mempool gate: verify signature and sufficient balance."""
    w = WALLETS_BY_ADDR.get(tx.get('sender', ''))
    if w is None or not verify_tx(tx, w):
        return False
    if led.balances.get(tx['sender'], 0) < tx['amount']:
        return False
    if tx['amount'] <= 0:
        return False
    pool.append(tx)
    return True


def mine_pending(led: ReserveLedger, ch: list[dict], pool: list[dict]) -> dict:
    txs = list(pool)
    pool.clear()
    block = mine_block(txs, ch[-1]['hash'], len(ch))
    for tx in txs:
        if 'sender' in tx and 'recipient' in tx:
            led.apply_transfer(tx['sender'], tx['recipient'], tx['amount'])
    ch.append(block)
    return block


tx3500 = build_transfer(sender, recipient.address, TRANSFER_AMOUNT)
assert verify_tx(tx3500, sender)
accepted = accept_tx(tx3500, ledger, mempool)
print('ACCEPT into mempool:', accepted, '| mempool size:', len(mempool))

block = mine_pending(ledger, chain, mempool)
print('MINED block', block['index'], 'hash', block['hash'][:16], '... nonce', block['nonce'])
print('BALANCES after mine — sender', ledger.balances[sender.address],
      '| recipient', ledger.balances[recipient.address],
      '| vault', ledger.vault_fiat)
assert ledger.vault_fiat == LOCAL_DEPOSIT  # transfer does not touch vault
assert ledger.invariant_ok()
assert verify_chain(chain)

ACCEPT into mempool: True | mempool size: 1
MINED block 1 hash 0067dcb85f4c95f3 ... nonce 187
BALANCES after mine — sender 1500 | recipient 3500 | vault 5000


## 8. Confirmation depth for payouts



In [36]:
def find_tx_block_index(ch: list[dict], predicate) -> int | None:
    for block in ch:
        for t in block['transactions']:
            if predicate(t):
                return block['index']
    return None


def confirmation_depth(ch: list[dict], inclusion_height: int) -> int:
    """k = H - h + 1 (inclusion counts as one confirmation)."""
    tip = ch[-1]['index']
    return tip - inclusion_height + 1


def ready_for_offramp(
    ch: list[dict],
    inclusion_height: int,
    min_confirmations: int = 1,
) -> bool:
    return confirmation_depth(ch, inclusion_height) >= min_confirmations


def mine_empty_for_depth(ch: list[dict], n: int = 1) -> None:
    """Mine empty (or marker) blocks to deepen confirmations — classroom only."""
    for i in range(n):
        marker = [{'note': f'confirmation-padding-{len(ch)}'}]
        ch.append(mine_block(marker, ch[-1]['hash'], len(ch)))


h_pay = find_tx_block_index(
    chain,
    lambda t: t.get('amount') == TRANSFER_AMOUNT and t.get('sender') == sender.address,
)
assert h_pay is not None
k_now = confirmation_depth(chain, h_pay)
print(f'Payment included at height h={h_pay}; tip H={chain[-1]["index"]}; depth k={k_now}')
print('ready_for_offramp (k_min=1):', ready_for_offramp(chain, h_pay, 1))

# Illustrate waiting for deeper confirmations (optional policy)
MIN_K_DEMO = 2
while not ready_for_offramp(chain, h_pay, MIN_K_DEMO):
    mine_empty_for_depth(chain, 1)
    print('  padded -> tip', chain[-1]['index'],
          'k=', confirmation_depth(chain, h_pay))

print(f'After padding: k={confirmation_depth(chain, h_pay)} >= {MIN_K_DEMO} -> payout allowed')
assert verify_chain(chain)

Payment included at height h=1; tip H=1; depth k=1
ready_for_offramp (k_min=1): True
  padded -> tip 2 k= 2
After padding: k=2 >= 2 -> payout allowed


## 9. Simulated off-ramp



In [37]:
assert ready_for_offramp(chain, h_pay, min_confirmations=1)

receipt = ledger.offramp(recipient.address, TRANSFER_AMOUNT)
assert ledger.invariant_ok()
assert ledger.vault_fiat == LOCAL_DEPOSIT - TRANSFER_AMOUNT  # 1500
assert ledger.balances.get(recipient.address, 0) == 0
assert ledger.balances[sender.address] == LOCAL_DEPOSIT - TRANSFER_AMOUNT

print('OFF-RAMP receipt:', receipt)
print('vault_fiat =', ledger.vault_fiat, '| circulating =', ledger.circulating)
print('invariant_ok =', ledger.invariant_ok())

OFF-RAMP receipt: {'receipt_id': '51ce87140d540fca', 'address': '56f62e486f82191a14f9f6ecaaa5329d0addfd02', 'paid_local': 3500}
vault_fiat = 1500 | circulating = 1500
invariant_ok = True


## 10. Balance 



In [38]:
def snapshot(led: ReserveLedger, label: str, s_addr: str, r_addr: str) -> dict:
    return {
        'step': label,
        'vault': led.vault_fiat,
        'circulating': led.circulating,
        'sender': led.balances.get(s_addr, 0),
        'recipient': led.balances.get(r_addr, 0),
        'invariant': led.invariant_ok(),
    }


# Fresh walk-through matching the Learning Guide numbers
demo_led = ReserveLedger()
s2, r2, _ = create_wallets()
rows = [snapshot(demo_led, 'start', s2.address, r2.address)]

demo_led.onramp(s2.address, 5000)
rows.append(snapshot(demo_led, 'on-ramp 5000', s2.address, r2.address))

demo_led.apply_transfer(s2.address, r2.address, 3500)
rows.append(snapshot(demo_led, 'transfer 3500', s2.address, r2.address))

demo_led.offramp(r2.address, 3500)
rows.append(snapshot(demo_led, 'off-ramp 3500', s2.address, r2.address))

print(f"{'step':<16} {'vault':>6} {'circ':>6} {'sender':>7} {'recip':>7} {'inv':>5}")
print('-' * 52)
for row in rows:
    print(f"{row['step']:<16} {row['vault']:6d} {row['circulating']:6d} "
          f"{row['sender']:7d} {row['recipient']:7d} {str(row['invariant']):>5}")

assert all(r['invariant'] for r in rows)
assert rows[-1]['vault'] == 1500 and rows[-1]['sender'] == 1500 and rows[-1]['recipient'] == 0
print('\nAccounting path OK')

step              vault   circ  sender   recip   inv
----------------------------------------------------
start                 0      0       0       0  True
on-ramp 5000       5000   5000    5000       0  True
transfer 3500      5000   5000    1500    3500  True
off-ramp 3500      1500   1500    1500       0  True

Accounting path OK


## 11. Reject invalid transfer



In [39]:
# Fresh mini-state so rejection is independent of prior balances
rej_led = ReserveLedger()
rej_led.onramp(sender.address, 5000)
rej_pool: list[dict] = []

bad = build_transfer(sender, recipient.address, 10)
bad['amount'] = 4999  # mutate after sign — signature no longer matches
tamper_rejected = not accept_tx(bad, rej_led, rej_pool)

overspend = build_transfer(sender, recipient.address, 10000)  # valid sig, insufficient funds
overspend_rejected = not accept_tx(overspend, rej_led, rej_pool)

print('tampered accept?', not tamper_rejected, '-> rejected?', tamper_rejected)
print('overspend accept?', not overspend_rejected, '-> rejected?', overspend_rejected)
print('mempool still empty?', len(rej_pool) == 0)
assert tamper_rejected and overspend_rejected and len(rej_pool) == 0
print('Invalid transfer rejection OK')

tampered accept? False -> rejected? True
overspend accept? False -> rejected? True
mempool still empty? True
Invalid transfer rejection OK


## 12. Explorer-style printout



In [40]:
def explorer_summary(ch: list[dict]) -> list[dict]:
    return [
        {
            'index': b['index'],
            'hash': b['hash'],
            'txs': b['transactions'],
        }
        for b in ch
    ]


def print_explorer(ch: list[dict]) -> None:
    for b in ch:
        print(f"Block {b['index']} {b['hash'][:12]}... txs={len(b['transactions'])}")
        for t in b['transactions']:
            if isinstance(t, dict) and 'amount' in t:
                print(f"    {t.get('sender', '?')[:8]}... -> {t.get('recipient', '?')[:8]}... "
                      f"amount={t['amount']} {t.get('asset', '')}")
            else:
                print('   ', t)


print_explorer(chain)
print('chain_ok =', verify_chain(chain))

Block 0 00a5c12dbd19... txs=1
    {'note': 'genesis'}
Block 1 0067dcb85f4c... txs=1
    cc6154ce... -> 56f62e48... amount=3500 mZAR
Block 2 00edf767904e... txs=1
    {'note': 'confirmation-padding-2'}
chain_ok = True


## 13. End-to-end remittance corridor demo



In [41]:
def run_remittance_demo(
    local_deposit: int = 5000,
    transfer_amount: int = 3500,
    min_confirmations: int = 1,
) -> dict:
    """Learning Guide remittance script — self-contained corridor demo."""
    snd, rcp, _rsv = create_wallets()
    led = ReserveLedger()
    ch: list[dict] = [mine_block([{'note': 'genesis'}], '0' * 64, 0)]

    # 1-2. On-ramp
    led.onramp(snd.address, local_deposit)
    assert led.invariant_ok()

    # 3. Signed transfer + mine
    tx = {
        'sender': snd.address,
        'recipient': rcp.address,
        'amount': transfer_amount,
        'timestamp': int(time.time()),
        'asset': 'mZAR',
    }
    tx['signature'] = sign(tx, snd.sk)
    assert verify_tx(tx, snd)

    block = mine_block([tx], ch[-1]['hash'], len(ch))
    ch.append(block)
    led.apply_transfer(snd.address, rcp.address, transfer_amount)

    # Confirmation depth gate (classroom default k_min=1)
    h = block['index']
    while confirmation_depth(ch, h) < min_confirmations:
        ch.append(mine_block([{'note': 'conf-pad'}], ch[-1]['hash'], len(ch)))
    assert ready_for_offramp(ch, h, min_confirmations)

    # 4. Off-ramp
    receipt = led.offramp(rcp.address, transfer_amount)
    assert led.invariant_ok()

    # 6. Invalid attempt (tamper after sign)
    bad = dict(tx)
    bad['amount'] = transfer_amount + 5000
    rejected = not verify_tx(bad, snd)

    return {
        'balances': dict(led.balances),
        'vault_fiat': led.vault_fiat,
        'circulating': led.circulating,
        'invariant': led.invariant_ok(),
        'chain_ok': verify_chain(ch),
        'receipt': receipt,
        'invalid_rejected': rejected,
        'confirmation_depth': confirmation_depth(ch, h),
        'payment_height': h,
        'explorer': explorer_summary(ch),
    }


out = run_remittance_demo()
summary = {k: out[k] for k in out if k != 'explorer'}
print(json.dumps(summary, indent=2))
print('blocks:', len(out['explorer']))

assert out['invariant'] is True
assert out['chain_ok'] is True
assert out['invalid_rejected'] is True
assert out['vault_fiat'] == 1500
assert out['circulating'] == 1500
print('\nEnd-to-end remittance corridor OK')

{
  "balances": {
    "cc6154ce75ed89ab8dea0b12e45866c988f513ef": 1500,
    "56f62e486f82191a14f9f6ecaaa5329d0addfd02": 0
  },
  "vault_fiat": 1500,
  "circulating": 1500,
  "invariant": true,
  "chain_ok": true,
  "receipt": {
    "receipt_id": "f3242c9e722cd958",
    "address": "56f62e486f82191a14f9f6ecaaa5329d0addfd02",
    "paid_local": 3500
  },
  "invalid_rejected": true,
  "confirmation_depth": 1,
  "payment_height": 1
}
blocks: 2

End-to-end remittance corridor OK


## 14. A9 MVP checklist wiring


In [42]:
def mvp_checklist(demo: dict) -> list[tuple[str, bool, str]]:
    """(letter, done?, evidence note) — A9 MVP from the slides."""
    has_transfer_tx = any(
        isinstance(t, dict) and t.get('amount') == 3500
        for b in demo['explorer']
        for t in b['txs']
    )
    return [
        ('(a) Wallets for parties', True, 'create_wallets() / Wallet dataclass'),
        ('(b) On-ramp at 1:1', demo['vault_fiat'] == 1500 and demo['invariant'], 'onramp then offramp left vault=1500'),
        ('(c) Signed transfer + mine', has_transfer_tx and demo['chain_ok'], '3500 mZAR in mined block'),
        ('(d) Off-ramp + receipt', 'receipt_id' in demo['receipt'], demo['receipt']['receipt_id']),
        ('(e) Verification (verify_chain + lookup)', demo['chain_ok'], f"payment_height={demo['payment_height']}"),
        ('(f) Payment API', False, 'optional Flask stretch below / wrap A8 routes'),
        ('(g) Explorer view', len(demo['explorer']) >= 2, f"{len(demo['explorer'])} blocks summarised"),
    ]


checks = mvp_checklist(out)
print('A9 MVP checklist (in-process path)')
print('-' * 72)
for label, done, note in checks:
    mark = '[x]' if done else '[ ]'
    print(f'  {mark} {label:<42} {note}')

core_ok = all(done for label, done, _ in checks if not label.startswith('(f)'))
assert core_ok
print('\nCore MVP items (a-e, g) satisfied in-process; (f) via optional Flask cell.')

A9 MVP checklist (in-process path)
------------------------------------------------------------------------
  [x] (a) Wallets for parties                    create_wallets() / Wallet dataclass
  [x] (b) On-ramp at 1:1                         onramp then offramp left vault=1500
  [x] (c) Signed transfer + mine                 3500 mZAR in mined block
  [x] (d) Off-ramp + receipt                     f3242c9e722cd958
  [x] (e) Verification (verify_chain + lookup)   payment_height=1
  [ ] (f) Payment API                            optional Flask stretch below / wrap A8 routes
  [x] (g) Explorer view                          2 blocks summarised

Core MVP items (a-e, g) satisfied in-process; (f) via optional Flask cell.


## 15. Optional stretch — Flask `test_client` remittance API



In [43]:
try:
    from flask import Flask, jsonify, request
except ImportError:
    Flask = None  # type: ignore
    print('Flask not installed — skip API stretch (pip install flask). In-process demo already complete.')


def register_remittance_routes(app: 'Flask', led: ReserveLedger, ch: list[dict], pool: list[dict],
                               wallets: dict[str, Wallet]) -> None:
    """Wire on-ramp / transfer / off-ramp / balances onto a Flask app."""

    @app.get('/')
    def index():
        return jsonify({
            'service': 'MiniRandCoin remittance (classroom)',
            'endpoints': [
                'POST /onramp', 'POST /transfer', 'POST /mine',
                'POST /offramp', 'GET /balances/<address>',
                'GET /chain', 'GET /explorer', 'POST /remit',
            ],
        })

    @app.post('/onramp')
    def api_onramp():
        body = request.get_json(force=True, silent=True) or {}
        address, amount = body.get('address'), body.get('amount')
        if not address or not isinstance(amount, int) or amount <= 0:
            return jsonify({'error': 'address and positive int amount required'}), 400
        led.onramp(address, amount)
        return jsonify({
            'event': 'onramp',
            'address': address,
            'amount': amount,
            'invariant': led.invariant_ok(),
            'vault_fiat': led.vault_fiat,
        }), 201

    @app.post('/transfer')
    def api_transfer():
        tx = request.get_json(force=True, silent=True) or {}
        w = wallets.get(tx.get('sender', ''))
        if w is None or not verify_tx(tx, w):
            return jsonify({'error': 'invalid signature or unknown sender'}), 400
        if led.balances.get(tx['sender'], 0) < tx.get('amount', 0):
            return jsonify({'error': 'insufficient balance'}), 400
        pool.append(tx)
        return jsonify({'accepted': True, 'mempool_size': len(pool)}), 201

    @app.post('/mine')
    def api_mine():
        if not ch:
            ch.append(mine_block([{'note': 'genesis'}], '0' * 64, 0))
        block = mine_pending(led, ch, pool)
        return jsonify({
            'index': block['index'],
            'hash': block['hash'],
            'transactions': block['transactions'],
            'invariant': led.invariant_ok(),
        }), 200

    @app.post('/offramp')
    def api_offramp():
        body = request.get_json(force=True, silent=True) or {}
        address, amount = body.get('address'), body.get('amount')
        if not address or not isinstance(amount, int) or amount <= 0:
            return jsonify({'error': 'address and positive int amount required'}), 400
        try:
            receipt = led.offramp(address, amount)
        except ValueError as exc:
            return jsonify({'error': str(exc)}), 400
        receipt = dict(receipt)
        receipt['invariant'] = led.invariant_ok()
        return jsonify(receipt), 200

    @app.get('/balances/<address>')
    def api_balances(address: str):
        return jsonify({
            'address': address,
            'balance': led.balances.get(address, 0),
            'vault_fiat': led.vault_fiat,
            'circulating': led.circulating,
            'invariant': led.invariant_ok(),
        })

    @app.get('/chain')
    def api_chain():
        return jsonify({'length': len(ch), 'chain': ch, 'chain_ok': verify_chain(ch)})

    @app.get('/explorer')
    def api_explorer():
        return jsonify(explorer_summary(ch))

    @app.post('/remit')
    def api_remit():
        body = request.get_json(force=True, silent=True) or {}
        deposit = int(body.get('local_deposit', 5000))
        amount = int(body.get('transfer_amount', 3500))
        # Reset in-process state for a clean orchestrated run
        led.vault_fiat = 0
        led.circulating = 0
        led.balances.clear()
        led.receipts.clear()
        pool.clear()
        ch.clear()
        ch.append(mine_block([{'note': 'genesis'}], '0' * 64, 0))

        snd = wallets['__sender__']
        rcp = wallets['__recipient__']
        led.onramp(snd.address, deposit)
        tx = build_transfer(snd, rcp.address, amount)
        pool.append(tx)
        mine_pending(led, ch, pool)
        receipt = led.offramp(rcp.address, amount)
        return jsonify({
            'vault_fiat': led.vault_fiat,
            'circulating': led.circulating,
            'balances': dict(led.balances),
            'invariant': led.invariant_ok(),
            'chain_ok': verify_chain(ch),
            'receipt': receipt,
            'explorer': explorer_summary(ch),
        }), 200


if Flask is not None:
    api_sender, api_recipient, api_reserve = create_wallets()
    api_ledger = ReserveLedger()
    api_chain: list[dict] = [mine_block([{'note': 'genesis'}], '0' * 64, 0)]
    api_pool: list[dict] = []
    api_wallets = {
        api_sender.address: api_sender,
        api_recipient.address: api_recipient,
        api_reserve.address: api_reserve,
        '__sender__': api_sender,
        '__recipient__': api_recipient,
    }

    app = Flask('mzar_remittance_demo')
    register_remittance_routes(app, api_ledger, api_chain, api_pool, api_wallets)
    client = app.test_client()

    r = client.get('/')
    assert r.status_code == 200

    r = client.post('/onramp', json={'address': api_sender.address, 'amount': 5000})
    assert r.status_code == 201 and r.get_json()['invariant'] is True

    pay = build_transfer(api_sender, api_recipient.address, 3500)
    r = client.post('/transfer', json=pay)
    assert r.status_code == 201

    # Reject tampered tx via API
    bad_api = dict(pay)
    bad_api['amount'] = 9999
    r = client.post('/transfer', json=bad_api)
    assert r.status_code == 400

    r = client.post('/mine')
    assert r.status_code == 200
    mined = r.get_json()
    print('API mined block', mined['index'], mined['hash'][:16], '...')

    r = client.get(f'/balances/{api_recipient.address}')
    assert r.get_json()['balance'] == 3500

    r = client.post('/offramp', json={'address': api_recipient.address, 'amount': 3500})
    assert r.status_code == 200
    assert r.get_json()['invariant'] is True
    assert api_ledger.vault_fiat == 1500

    r = client.get('/explorer')
    assert r.status_code == 200 and len(r.get_json()) >= 2

    r = client.post('/remit', json={'local_deposit': 5000, 'transfer_amount': 3500})
    assert r.status_code == 200
    remit = r.get_json()
    assert remit['invariant'] and remit['chain_ok'] and remit['vault_fiat'] == 1500

    print('Flask test_client remittance API OK')
    print('  vault after /remit:', remit['vault_fiat'], '| receipt:', remit['receipt']['receipt_id'])
else:
    print('(f) Payment API left as A9 homework if Flask unavailable')

API mined block 1 0067dcb85f4c95f3 ...
Flask test_client remittance API OK
  vault after /remit: 1500 | receipt: 8d09a3939271a0d5


## 16. Session wrap



In [44]:
# Final smoke — re-run the Learning Guide script once more
final = run_remittance_demo()
assert final['invariant'] and final['chain_ok'] and final['invalid_rejected']
assert final['vault_fiat'] == 1500
print('Lecture 10 live-coding complete.')
print(json.dumps(
    {k: final[k] for k in ('vault_fiat', 'circulating', 'invariant', 'chain_ok', 'invalid_rejected')},
    indent=2,
))

Lecture 10 live-coding complete.
{
  "vault_fiat": 1500,
  "circulating": 1500,
  "invariant": true,
  "chain_ok": true,
  "invalid_rejected": true
}
